#### Classification – High-Demand Session Prediction
Here I'll classify each session as High Demand (Demand ≥ train median) or Normal Demand, compare multiple models and explain the best one.

#### Step-0: Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import TimeSeriesSplit, cross_validate, RandomizedSearchCV

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay

import shap
from lime.lime_tabular import LimeTabularExplainer

#### Step-1: Load and time-based split

In [2]:
# Same split as regression: first 80% of sessions for train, last 20% for test
df= pd.read_csv("../data/cleaned/model_ready.csv", parse_dates=['StartDatetime'])
split= int(len(df)*0.8)
train= df.iloc[:split].copy()
test= df.iloc[split:].copy()
train.shape, test.shape

((57670, 14), (14418, 14))

#### Step-2: Creating the target

In [3]:
# Median from TRAIN set only, so test data does not leak.
median_demand= train['Demand'].median()
median_demand

np.float64(14.35)